In [16]:
import pandas as pd
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn import datasets as ds
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import decomposition

from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sklearn.metrics as sklm
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import plot_tree
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report, average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report, average_precision_score, precision_recall_curve, roc_auc_score

from scipy.stats import ttest_ind

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

In [17]:
with zipfile.ZipFile("ecg_data.zip","r") as zip_ref:
    zip_ref.extractall("ecg_data")

def load_data():
    this_directory = os.getcwd()
    data = pd.read_csv(os.path.join(this_directory, 'ecg_data/ecg_data.csv'), index_col=0)
    return data

raw_data = load_data()

In [8]:
print(f'The number of samples: {len(raw_data.index)}')
print(f'The number of columns: {len(raw_data.columns)}')

print(f'The number of NaN values in the entire dataframe: {raw_data.isnull().sum().sum()}')
print(f'The number of samples with label 0: {len(raw_data[raw_data["label"] == 0])}')
print(f'The number of samples with label 1: {len(raw_data[raw_data["label"] == 1])}')
print(f'The percentage of samples with label 0 is thus {len(raw_data[raw_data["label"] == 0])/len(raw_data.index)*100:.2f}%', 
      f'and the percentage with label 1 {len(raw_data[raw_data["label"] == 1])/len(raw_data.index)*100:.2f}%')

# print(raw_data.groupby('label').count())
# print(raw_data.groupby('label').mean())
# print(raw_data.groupby('label').var())
# print(raw_data.groupby('label').std())

The number of samples: 827
The number of columns: 9001
The number of NaN values in the entire dataframe: 0
The number of samples with label 0: 681
The number of samples with label 1: 146
The percentage of samples with label 0 is thus 82.35% and the percentage with label 1 17.65%


In [18]:
X = raw_data.drop('label', axis=1)
Y = raw_data['label']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=4, stratify=Y)

In [19]:
class VarianceFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.columns_to_drop_ = [col for col in X.columns if np.var(X[col]) < self.threshold]
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.columns_to_drop_ = [col for col in upper.columns if any(upper[col] > self.threshold)]
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class TTestFilter(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=0.05):
        self.alpha = alpha

    def fit(self, X, y):
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]

        self.columns_to_drop_ = []
        threshold = self.alpha / X.shape[1]  # Bonferroni

        for col in X.columns:
            group0 = X.loc[y == 0, col]
            group1 = X.loc[y == 1, col]
            _, p_value = ttest_ind(group0, group1, nan_policy="omit")

            if p_value > threshold:
                self.columns_to_drop_.append(col)

        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


pipeline = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ("scaler", RobustScaler())
    ])

pipeline.fit(x_train, y_train)
x_train_preprocessed = pipeline.transform(x_train)
x_test_preprocessed = pipeline.transform(x_test)

#Amount of features after preprocessing 
x_train_var = pipeline.named_steps["variance"].transform(x_train)
x_train_corr = pipeline.named_steps["correlation"].transform(x_train_var)
x_train_t = pipeline.named_steps["ttest"].transform(x_train_corr)

print(f"Start amount features: {x_train.shape[1]}")
print(f"After the variance filter: {x_train_var.shape[1]}")
print(f"After the correlation filter: {x_train_corr.shape[1]}")
print(f"After the t-test filter: {x_train_t.shape[1]}")

Start amount features: 9000
After the variance filter: 9000
After the correlation filter: 4068
After the t-test filter: 47


In [20]:
def evaluate_model_on_test(clf, X_test, y_test, name="Model"):
    y_pred = clf.predict(X_test)
    if hasattr(clf, 'predict_proba'):
        y_score = clf.predict_proba(X_test)[:, 1]
    else:
        y_score = clf.decision_function(X_test)

    # Calculate metrics
    precision = precision_score(y_test, y_pred, zero_division=1)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test, y_score)
    auc_roc = roc_auc_score(y_test, y_score)

    # Print resultats
    print(f"--- Evaluation: {name} ---")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"F1-Score:   {f1:.4f}")
    print(f"AUC-PR:     {auc_pr:.4f}")
    print(f"ROC-AUC:    {auc_roc:.4f}")
    print("-" * 30)


    ## How to call to function: (example for SVM)
    ## evaluate_model_on_test(best_svm, x_test, y_test, name="Optimized SVM")

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from tempfile import mkdtemp
from shutil import rmtree

# 1. Cache instellen voor snelheid tijdens tuning
cachedir = mkdtemp()

# 2. De nieuwe Pipeline met jouw specifieke klassen
svm_pipeline = Pipeline([
    ('variance', VarianceFilter(threshold=0.01)),
    ('correlation', CorrelationFilter(threshold=0.95)),
    ('ttest', TTestFilter(alpha=0.05)),
    ('scaler', RobustScaler()),
    ('svm', SVC(probability=True, class_weight='balanced', random_state=4))
], memory=cachedir)

# 3. Hyperparameter grids (ongewijzigd, maar gekoppeld aan de nieuwe pipeline)
param_distributions = [
    {
        'svm__kernel': ['rbf', 'sigmoid'],
        'svm__C': [0.01, 0.1, 1, 10, 100],
        'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1]
    },
    {
        'svm__kernel': ['poly'],
        'svm__C': [0.01, 0.1, 1, 10],
        'svm__degree': [2, 3, 4]
    },
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.01, 0.1, 1, 10, 100]
    }
]

# 4. RandomizedSearchCV configuratie
random_search_svm = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=param_distributions,
    n_iter=50, 
    scoring='average_precision', 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=4),
    verbose=2,
    n_jobs=1 
)

# 5. Training
print("Start training met de nieuwe preprocessing pipeline...")
random_search_svm.fit(x_train, y_train)

# 6. Cleanup cache
try:
    rmtree(cachedir)
except:
    pass

print(f"Beste instellingen gevonden: {random_search_svm.best_params_}")

Start training met de nieuwe preprocessing pipeline...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END ......svm__C=100, svm__gamma=scale, svm__kernel=rbf; total time= 2.5min
[CV] END ......svm__C=100, svm__gamma=scale, svm__kernel=rbf; total time= 2.3min
[CV] END ......svm__C=100, svm__gamma=scale, svm__kernel=rbf; total time= 2.4min
[CV] END ......svm__C=100, svm__gamma=scale, svm__kernel=rbf; total time= 2.4min
[CV] END ......svm__C=100, svm__gamma=scale, svm__kernel=rbf; total time= 2.3min
[CV] END ........svm__C=1, svm__gamma=scale, svm__kernel=rbf; total time=   0.7s
[CV] END ........svm__C=1, svm__gamma=scale, svm__kernel=rbf; total time=   0.7s
[CV] END ........svm__C=1, svm__gamma=scale, svm__kernel=rbf; total time=   0.7s
[CV] END ........svm__C=1, svm__gamma=scale, svm__kernel=rbf; total time=   0.6s
[CV] END ........svm__C=1, svm__gamma=scale, svm__kernel=rbf; total time=   0.6s
[CV] END .....svm__C=10, svm__gamma=0.1, svm__kernel=sigmoid; total time=

In [25]:
best_svm_model = random_search_svm.best_estimator_
evaluate_model_on_test(best_svm_model, x_test, y_test, name="Optimized SVM Pipeline")

--- Evaluation: Optimized SVM Pipeline ---
Precision:  0.2750
Recall:     0.7586
F1-Score:   0.4037
AUC-PR:     0.4827
ROC-AUC:    0.7367
------------------------------
